## Install Library

In [1]:
# Install the YOLOv8 package
!pip install ultralytics

# Clear the output and check the GPU status
import ultralytics
ultralytics.checks()

Ultralytics 8.4.219 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 0.1/225.8 GB disk)


## Mount with Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load Dataset

In [ ]:
# Navigate to your project folder
%cd /content/drive/MyDrive/Thesis_Pothole_Project

# Unzip the file into a new folder called Dataset_1_Baseline
!unzip "Pothole detection lowlight.v1i.yolov8.zip" -d Dataset_1_Baseline

Streaming output truncated to the last 5000 lines.
  inflating: Dataset_1_Baseline/valid/labels/2445_jpg.rf.e870db38523d8d1e1de80cb395c2ce5d.txt  
  inflating: Dataset_1_Baseline/valid/labels/2860_jpg.rf.ed3963d0138a5ead0daaeb17ebca0ecd.txt  
  inflating: Dataset_1_Baseline/valid/labels/img-516_jpg.rf.6c6aa22cab9a3e66c548e697303738f8.txt  
  inflating: Dataset_1_Baseline/valid/labels/potholes491_png.rf.2ee36bd5e4025cdd5762eec9c6f48990.txt  
  inflating: Dataset_1_Baseline/valid/labels/ddpi13_file435_jpg.rf.67b400e975aa6668343c1c633e2a9d94.txt  
  inflating: Dataset_1_Baseline/valid/labels/712_jpg.rf.7bbf5c03eb0bd06a1d76adbe46b1bcfd.txt  
  inflating: Dataset_1_Baseline/valid/labels/1233_jpg.rf.6a52ca5de368057e503693c55443a94c.txt  
  inflating: Dataset_1_Baseline/valid/labels/770_jpg.rf.5d660b27e8105243172c5188cd0b5e7a.txt  
  inflating: Dataset_1_Baseline/valid/labels/3486_jpg.rf.9b2a6c65baa39952039481aac2ae5dfe.txt  
  inflating: Dataset_1_Baseline/valid/labels/2863_jpg.rf.0efccbfd16

## Image Generation For Training, Validation and Test/Night

In [ ]:
import cv2
import os
import shutil
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. PURE MATHEMATICAL ALGORITHMS
# ==========================================e

def apply_clahe(img, clip_limit=2.0, grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

def apply_agcwd(img, w=0.5):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    v_channel = hsv[:, :, 2]
    hist, _ = np.histogram(v_channel.flatten(), 256, [0, 256])
    pdf = hist / np.sum(hist)
    max_pdf = np.max(pdf)
    min_pdf = np.min(pdf)
    if max_pdf == min_pdf: return img
    w_pdf = max_pdf * (((pdf - min_pdf) / (max_pdf - min_pdf)) ** w)
    w_cdf = np.cumsum(w_pdf) / np.sum(w_pdf)
    mapping = np.array([255 * (i / 255) ** (1 - w_cdf[i]) for i in range(256)], dtype=np.uint8)
    hsv[:, :, 2] = cv2.LUT(v_channel, mapping)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def apply_ssr(img, sigma=50):
    img_float = np.float64(img) + 1.0
    illumination = cv2.GaussianBlur(img_float, (0, 0), sigma)
    retinex = np.log10(img_float) - np.log10(illumination)
    enhanced = np.zeros_like(img)
    for i in range(3):
        channel = retinex[:, :, i]
        lower, upper = np.percentile(channel, (1, 99))
        channel_clipped = np.clip(channel, lower, upper)
        channel_norm = (channel_clipped - lower) / (upper - lower) * 255.0
        enhanced[:, :, i] = np.uint8(channel_norm)
    return enhanced

# ==========================================
# 2. METHODOLOGY-SAFE DATASET GENERATOR
# ==========================================

def generate_experimental_dataset(source_base, target_base, algo_func, algo_name, splits=['train', 'valid', 'dataset_split/test_night']):
    print(f"\n🚀 Generating {algo_name} Dataset...")

    for split in splits:
        source_images = os.path.join(source_base, split, 'images')
        source_labels = os.path.join(source_base, split, 'labels')
        target_images = os.path.join(target_base, split, 'images')
        target_labels = os.path.join(target_base, split, 'labels')

        os.makedirs(target_images, exist_ok=True)
        # Removed: os.makedirs(target_labels, exist_ok=True) # Moved this line to be closer to where labels are copied

        # 1. Process Images with TQDM Progress Bar
        if os.path.exists(source_images):
            image_files = [f for f in os.listdir(source_images) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            # --> TQDM added here for a clean progress bar! <--
            for img_name in tqdm(image_files, desc=f"Processing {split}"):
                img_path = os.path.join(source_images, img_name)
                img = cv2.imread(img_path)
                if img is None: continue

                enhanced_img = algo_func(img)
                cv2.imwrite(os.path.join(target_images, img_name), enhanced_img)

        # 2. Copy Labels Safely
        if os.path.exists(source_labels):
            os.makedirs(target_labels, exist_ok=True) # Ensure target label directory exists right before copying labels
            for label_file in os.listdir(source_labels):
                if label_file.endswith('.txt'):
                    shutil.copy2(
                        os.path.join(source_labels, label_file),
                        os.path.join(target_labels, label_file)
                    )

# ==========================================
# 3. RUN THE GENERATOR
# ==========================================
SOURCE = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline"

generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '2_CLAHE')}", apply_clahe, "CLAHE")
generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '3_AGCWD')}", apply_agcwd, "AGCWD")
generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '4_SSR')}", apply_ssr, "SSR")

print("\n🎉 ALL EXPERIMENTAL DATASETS GENERATED SUCCESSFULLY!")


🚀 Generating CLAHE Dataset...


Processing dataset_split/test_night: 100%|██████████| 289/289 [00:09<00:00, 30.91it/s]



🚀 Generating AGCWD Dataset...


Processing dataset_split/test_night: 100%|██████████| 289/289 [00:10<00:00, 27.89it/s]



🚀 Generating SSR Dataset...


Processing dataset_split/test_night: 100%|██████████| 289/289 [02:30<00:00,  1.92it/s]



🎉 ALL EXPERIMENTAL DATASETS GENERATED SUCCESSFULLY!


## Image Generation For Test/Day

In [ ]:
import cv2
import os
import shutil
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. PURE MATHEMATICAL ALGORITHMS
# ==========================================

def apply_clahe(img, clip_limit=2.0, grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

def apply_agcwd(img, w=0.5):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    v_channel = hsv[:, :, 2]
    hist, _ = np.histogram(v_channel.flatten(), 256, [0, 256])
    pdf = hist / np.sum(hist)
    max_pdf = np.max(pdf)
    min_pdf = np.min(pdf)
    if max_pdf == min_pdf: return img
    w_pdf = max_pdf * (((pdf - min_pdf) / (max_pdf - min_pdf)) ** w)
    w_cdf = np.cumsum(w_pdf) / np.sum(w_pdf)
    mapping = np.array([255 * (i / 255) ** (1 - w_cdf[i]) for i in range(256)], dtype=np.uint8)
    hsv[:, :, 2] = cv2.LUT(v_channel, mapping)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def apply_ssr(img, sigma=50):
    img_float = np.float64(img) + 1.0
    illumination = cv2.GaussianBlur(img_float, (0, 0), sigma)
    retinex = np.log10(img_float) - np.log10(illumination)
    enhanced = np.zeros_like(img)
    for i in range(3):
        channel = retinex[:, :, i]
        lower, upper = np.percentile(channel, (1, 99))
        channel_clipped = np.clip(channel, lower, upper)
        channel_norm = (channel_clipped - lower) / (upper - lower) * 255.0
        enhanced[:, :, i] = np.uint8(channel_norm)
    return enhanced

# ==========================================
# 2. METHODOLOGY-SAFE DATASET GENERATOR
# ==========================================

def generate_experimental_dataset(source_base, target_base, algo_func, algo_name, splits=['dataset_split/test_day']):
    print(f"\n🚀 Generating {algo_name} Dataset for {splits}...")

    for split in splits:
        source_images = os.path.join(source_base, split, 'images')
        source_labels = os.path.join(source_base, split, 'labels')
        target_images = os.path.join(target_base, split, 'images')
        target_labels = os.path.join(target_base, split, 'labels')

        os.makedirs(target_images, exist_ok=True)

        # 1. Process Images with TQDM Progress Bar
        if os.path.exists(source_images):
            image_files = [f for f in os.listdir(source_images) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            for img_name in tqdm(image_files, desc=f"Processing {split}"):
                img_path = os.path.join(source_images, img_name)
                img = cv2.imread(img_path)
                if img is None: continue

                enhanced_img = algo_func(img)
                cv2.imwrite(os.path.join(target_images, img_name), enhanced_img)

        # 2. Copy Labels Safely
        if os.path.exists(source_labels):
            os.makedirs(target_labels, exist_ok=True)
            for label_file in os.listdir(source_labels):
                if label_file.endswith('.txt'):
                    shutil.copy2(
                        os.path.join(source_labels, label_file),
                        os.path.join(target_labels, label_file)
                    )

# ==========================================
# 3. RUN THE GENERATOR
# ==========================================
SOURCE = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline"
TARGET_SPLITS = ['dataset_split/test_day']

generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '2_CLAHE')}", apply_clahe, "CLAHE", splits=TARGET_SPLITS)
generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '3_AGCWD')}", apply_agcwd, "AGCWD", splits=TARGET_SPLITS)
generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '4_SSR')}", apply_ssr, "SSR", splits=TARGET_SPLITS)

print("\n🎉 TEST_DAY EXPERIMENTAL DATASETS GENERATED SUCCESSFULLY!")


🚀 Generating CLAHE Dataset for ['dataset_split/test_day']...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:12<00:00, 22.63it/s]



🚀 Generating AGCWD Dataset for ['dataset_split/test_day']...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:10<00:00, 27.02it/s]



🚀 Generating SSR Dataset for ['dataset_split/test_day']...


Processing dataset_split/test_day: 100%|██████████| 294/294 [02:34<00:00,  1.91it/s]



🎉 TEST_DAY EXPERIMENTAL DATASETS GENERATED SUCCESSFULLY!


## Proxy Judge to find best parameters

In [ ]:
import cv2
import os
import yaml
import shutil
import itertools
import numpy as np
import pandas as pd
from functools import partial
from tqdm import tqdm
from ultralytics import YOLO

# ==========================================
# 1. PURE MATHEMATICAL ALGORITHMS
# ==========================================

def apply_clahe(img, clip_limit=2.0, grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

def apply_agcwd(img, w=0.5):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    v_channel = hsv[:, :, 2]
    hist, _ = np.histogram(v_channel.flatten(), 256, [0, 256])
    pdf = hist / np.sum(hist)
    max_pdf = np.max(pdf)
    min_pdf = np.min(pdf)
    if max_pdf == min_pdf:
        return img
    w_pdf = max_pdf * (((pdf - min_pdf) / (max_pdf - min_pdf)) ** w)
    w_cdf = np.cumsum(w_pdf) / np.sum(w_pdf)
    mapping = np.array([255.0 * ((i / 255.0) ** (1.0 - w_cdf[i])) for i in range(256)], dtype=np.uint8)
    hsv[:, :, 2] = cv2.LUT(v_channel, mapping)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def apply_ssr(img, sigma=50):
    img_float = np.float64(img) + 1.0
    illumination = cv2.GaussianBlur(img_float, (0, 0), sigma)
    retinex = np.log10(img_float) - np.log10(illumination)
    enhanced = np.zeros_like(img)
    for i in range(3):
        channel = retinex[:, :, i]
        lower, upper = np.percentile(channel, (1, 99))
        if upper == lower:
            continue
        channel_clipped = np.clip(channel, lower, upper)
        channel_norm = ((channel_clipped - lower) / (upper - lower)) * 255.0
        enhanced[:, :, i] = np.uint8(channel_norm)
    return enhanced

# ==========================================
# 2. YOLOv8 mAP50 GRID SEARCH ENGINE
# ==========================================

def run_yolo_grid_search(source_base, yolo_weights_path):
    """
    Evaluates every hyperparameter combination on the 'valid' split using YOLOv8 mAP50.
    Uses Colab local storage (/content/temp_grid_eval) for maximum speed.
    """
    print(f"\nLoading Baseline YOLOv8 Model: {yolo_weights_path}")
    model = YOLO(yolo_weights_path)

    valid_src_images = os.path.join(source_base, 'valid', 'images')
    valid_src_labels = os.path.join(source_base, 'valid', 'labels')

    # Setup fast local temporary directory on Colab SSD
    temp_base = "/content/temp_grid_eval"
    temp_images = os.path.join(temp_base, "valid", "images")
    temp_labels = os.path.join(temp_base, "valid", "labels")

    if os.path.exists(temp_base):
        shutil.rmtree(temp_base)
    os.makedirs(temp_images, exist_ok=True)
    os.makedirs(temp_labels, exist_ok=True)

    # Copy validation labels once to local SSD
    for lbl in os.listdir(valid_src_labels):
        if lbl.endswith('.txt'):
            shutil.copy2(os.path.join(valid_src_labels, lbl), os.path.join(temp_labels, lbl))

    # Create temporary data.yaml using class names directly from the trained model
    temp_yaml_path = os.path.join(temp_base, "temp_grid_data.yaml")
    yaml_content = {
        'path': temp_base,
        'train': 'valid/images',
        'val': 'valid/images',
        'names': model.names
    }
    with open(temp_yaml_path, 'w') as f:
        yaml.dump(yaml_content, f)

    image_files = [f for f in os.listdir(valid_src_images) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    # Define Parameter Grids
    grids = {
        "CLAHE": {
            "func": apply_clahe,
            "param_names": ["clip_limit", "grid_size"],
            "combinations": list(itertools.product([1.0, 2.0, 3.0, 4.0], [(4, 4), (8, 8), (16, 16)]))
        },
        "AGCWD": {
            "func": apply_agcwd,
            "param_names": ["w"],
            "combinations": [(w,) for w in [0.1, 0.3, 0.5, 0.7, 0.9]]
        },
        "SSR": {
            "func": apply_ssr,
            "param_names": ["sigma"],
            "combinations": [(s,) for s in [15, 50, 80, 120, 250]]
        }
    }

    best_params = {}
    all_results = []

    print(f"\n🔍 Starting YOLOv8 mAP50 Grid Search on {len(image_files)} validation images...")

    for algo_name, config in grids.items():
        print(f"\n================ Tuning {algo_name} ({len(config['combinations'])} combinations) ================")
        best_map50 = -1.0
        best_map50_95 = -1.0
        best_combo = None

        for combo in config["combinations"]:
            params_dict = dict(zip(config["param_names"], combo))

            # 1. Enhance validation images and write to fast local folder
            for img_name in tqdm(image_files, desc=f"Enhancing {algo_name} {params_dict}", leave=False):
                img_path = os.path.join(valid_src_images, img_name)
                img = cv2.imread(img_path)
                if img is None:
                    continue
                enhanced = config["func"](img, **params_dict)
                cv2.imwrite(os.path.join(temp_images, img_name), enhanced)

            # Remove YOLO label cache if it exists to ensure fresh evaluation
            cache_file = os.path.join(temp_base, "valid", "labels.cache")
            if os.path.exists(cache_file):
                os.remove(cache_file)

            # 2. Run YOLOv8 Validation
            metrics = model.val(
                data=temp_yaml_path,
                split='val',
                verbose=False,
                plots=False
            )

            map50 = float(metrics.box.map50)
            map50_95 = float(metrics.box.map)
            precision = float(metrics.box.mp)
            recall = float(metrics.box.mr)

            print(f"   Params: {params_dict} | P: {precision:.4f} | R: {recall:.4f} | mAP50: {map50:.4f} | mAP50-95: {map50_95:.4f}")

            all_results.append({
                "Algorithm": algo_name,
                "Parameters": str(params_dict),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "mAP50": round(map50, 4),
                "mAP50_95": round(map50_95, 4)
            })

            # Pick highest mAP50 (use mAP50-95 for tie situation)
            if (map50 > best_map50) or (map50 == best_map50 and map50_95 > best_map50_95):
                best_map50 = map50
                best_map50_95 = map50_95
                best_combo = params_dict

        best_params[algo_name] = best_combo
        print(f"Best {algo_name} Parameters: {best_combo} (mAP50: {best_map50:.4f}, mAP50-95: {best_map50_95:.4f})")

    # Save Grid Search Table to CSV
    results_df = pd.DataFrame(all_results)
    csv_path = os.path.join(os.path.dirname(source_base), "Grid_Search_YOLO_mAP_Results.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"\nSaved full YOLOv8 Grid Search table to: {csv_path}")

    # Clean up temporary local folder
    shutil.rmtree(temp_base, ignore_errors=True)

    return best_params

# ==========================================
# 3. METHODOLOGY-SAFE DATASET GENERATOR
# ==========================================

def generate_experimental_dataset(
    source_base,
    target_base,
    algo_func,
    algo_name,
    splits=['train', 'valid', 'dataset_split/test_night', 'dataset_split/test_day']
):
    print(f"\nGenerating {algo_name} Dataset across {len(splits)} splits...")

    for split in splits:
        source_images = os.path.join(source_base, split, 'images')
        source_labels = os.path.join(source_base, split, 'labels')
        target_images = os.path.join(target_base, split, 'images')
        target_labels = os.path.join(target_base, split, 'labels')

        # 1. Process Images
        if os.path.exists(source_images):
            os.makedirs(target_images, exist_ok=True)
            image_files = [f for f in os.listdir(source_images) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            for img_name in tqdm(image_files, desc=f"Processing {split}"):
                img_path = os.path.join(source_images, img_name)
                img = cv2.imread(img_path)
                if img is None:
                    continue

                enhanced_img = algo_func(img)
                cv2.imwrite(os.path.join(target_images, img_name), enhanced_img)
        else:
            print(f"Warning: Split folder not found and skipped -> {source_images}")

        # 2. Copy Labels
        if os.path.exists(source_labels):
            os.makedirs(target_labels, exist_ok=True)
            for label_file in os.listdir(source_labels):
                if label_file.endswith('.txt'):
                    shutil.copy2(
                        os.path.join(source_labels, label_file),
                        os.path.join(target_labels, label_file)
                    )

# ==========================================
# 4. RUN GRID SEARCH & GENERATE DATASETS
# ==========================================

SOURCE = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline"

BASELINE_WEIGHTS = "/content/drive/MyDrive/Thesis_Pothole_Project/Training_Results/Optimized_Model_Baseline2_YOLOv8n/weights/best.pt"

# 1. Find the best hyperparameters on the valid split using mAP50
best_hyperparams = run_yolo_grid_search(
    source_base=SOURCE,
    yolo_weights_path=BASELINE_WEIGHTS
)

# 2. Lock in the winning hyperparameters for each algorithm
optimal_clahe = partial(apply_clahe, **best_hyperparams["CLAHE"])
optimal_agcwd = partial(apply_agcwd, **best_hyperparams["AGCWD"])
optimal_ssr   = partial(apply_ssr,   **best_hyperparams["SSR"])

# 3.: Generate the 3 final datasets
ALL_SPLITS = ['train', 'valid', 'dataset_split/test_night', 'dataset_split/test_day']

generate_experimental_dataset(
    SOURCE,
    SOURCE.replace('1_Baseline', '2_CLAHE_Tuned'),
    optimal_clahe,
    f"CLAHE {best_hyperparams['CLAHE']}",
    splits=ALL_SPLITS
)

generate_experimental_dataset(
    SOURCE,
    SOURCE.replace('1_Baseline', '3_AGCWD_Tuned'),
    optimal_agcwd,
    f"AGCWD {best_hyperparams['AGCWD']}",
    splits=ALL_SPLITS
)

generate_experimental_dataset(
    SOURCE,
    SOURCE.replace('1_Baseline', '4_SSR_Tuned'),
    optimal_ssr,
    f"SSR {best_hyperparams['SSR']}",
    splits=ALL_SPLITS
)

print("\nYOLOv8 mAP50 GRID SEARCH & ALL 4 SPLITS GENERATED SUCCESSFULLY!")


Loading Baseline YOLOv8 Model: /content/drive/MyDrive/Thesis_Pothole_Project/Training_Results/Optimized_Model_Baseline2_YOLOv8n/weights/best.pt

🔍 Starting YOLOv8 mAP50 Grid Search on 1242 validation images...

================ Tuning CLAHE (12 combinations) ================


Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2530.3±477.5 MB/s, size: 141.8 KB)
val: Scanning /content/temp_grid_eval/valid/labels... 1242 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1242/1242 2.0Kit/s 0.6s
val: New cache created: /content/temp_grid_eval/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 78/78 7.6it/s 10.3s
                   all       1242       3069      0.849      0.739      0.813      0.496
Speed: 0.9ms preprocess, 4.0ms inference, 0.0ms loss, 1.1ms postprocess per image
   Params: {'clip_limit': 1.0, 'grid_size': (4, 4)} | P: 0.8489 | R: 0.7393 | mAP50: 0.8128 | mAP50-95: 0.4960


Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2553.6±1332.9 MB/s, size: 137.1 KB)
val: Scanning /content/temp_grid_eval/valid/labels... 1242 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1242/1242 2.1Kit/s 0.6s
val: New cache created: /content/temp_grid_eval/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 78/78 7.6it/s 10.2s
                   all       1242       3069      0.841      0.752      0.815      0.497
Speed: 1.0ms preprocess, 3.9ms inference, 0.0ms loss, 1.2ms postprocess per image
   Params: {'clip_limit': 1.0, 'grid_size': (8, 8)} | P: 0.8406 | R: 0.7520 | mAP50: 0.8150 | mAP50-95: 0.4974


Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)


Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2944.4±738.3 MB/s, size: 103.1 KB)
val: Scanning /content/temp_grid_eval/valid/labels... 1242 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1242/1242 2.2Kit/s 0.6s
val: New cache created: /content/temp_grid_eval/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 78/78 7.6it/s 10.3s
                   all       1242       3069      0.829      0.752      0.814      0.494
Speed: 1.0ms preprocess, 3.9ms inference, 0.0ms loss, 1.1ms postprocess per image
   Params: {'clip_limit': 1.0, 'grid_size': (16, 16)} | P: 0.8291 | R: 0.7522 | mAP50: 0.8144 | mAP50-95: 0.4944


Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


... (34 outputs left out)

Processing dataset_split/test_day: 100%|██████████| 294/294 [00:09<00:00, 29.49it/s]



Generating SSR {'sigma': 50} Dataset across 4 splits...


Processing dataset_split/test_day: 100%|██████████| 294/294 [02:18<00:00,  2.12it/s]



YOLOv8 mAP50 GRID SEARCH & ALL 4 SPLITS GENERATED SUCCESSFULLY!


## Micro Training Grid Search

In [ ]:
import cv2
import os
import shutil
import random
import yaml
import itertools
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO

# ==========================================
# 1. PURE MATHEMATICAL ALGORITHMS
# ==========================================
def apply_clahe(img, clip_limit=2.0, grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

def apply_agcwd(img, w=0.5):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    v_channel = hsv[:, :, 2]
    hist, _ = np.histogram(v_channel.flatten(), 256, [0, 256])
    pdf = hist / np.sum(hist)
    max_pdf = np.max(pdf)
    min_pdf = np.min(pdf)
    if max_pdf == min_pdf: return img
    w_pdf = max_pdf * (((pdf - min_pdf) / (max_pdf - min_pdf)) ** w)
    w_cdf = np.cumsum(w_pdf) / np.sum(w_pdf)
    mapping = np.array([255.0 * ((i / 255.0) ** (1.0 - w_cdf[i])) for i in range(256)], dtype=np.uint8)
    hsv[:, :, 2] = cv2.LUT(v_channel, mapping)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def apply_ssr(img, sigma=50):
    img_float = np.float64(img) + 1.0
    illumination = cv2.GaussianBlur(img_float, (0, 0), sigma)
    retinex = np.log10(img_float) - np.log10(illumination)
    enhanced = np.zeros_like(img)
    for i in range(3):
        channel = retinex[:, :, i]
        lower, upper = np.percentile(channel, (1, 99))
        if upper == lower: continue
        channel_clipped = np.clip(channel, lower, upper)
        channel_norm = ((channel_clipped - lower) / (upper - lower)) * 255.0
        enhanced[:, :, i] = np.uint8(channel_norm)
    return enhanced

# ==========================================
# 2. MICRO-DATASET SAMPLER
# ==========================================
def create_micro_dataset(source_base, temp_base, sample_ratio=0.2):
    # Randomly copies a small percentage of images/labels to a temporary local folder for fast training
    print(f"\nCreating {sample_ratio*100}% Micro-Dataset for fast tuning...")
    if os.path.exists(temp_base):
        shutil.rmtree(temp_base)

    selected_files = {}

    for split in ['train', 'valid']:
        src_img_dir = os.path.join(source_base, split, 'images')
        src_lbl_dir = os.path.join(source_base, split, 'labels')

        all_imgs = [f for f in os.listdir(src_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
        sample_size = max(1, int(len(all_imgs) * sample_ratio))

        # Randomly select subset
        random.seed(42) # Ensure repeatability
        sampled_imgs = random.sample(all_imgs, sample_size)
        selected_files[split] = sampled_imgs

        tgt_lbl_dir = os.path.join(temp_base, split, 'labels')
        os.makedirs(tgt_lbl_dir, exist_ok=True)

        for img_name in tqdm(sampled_imgs, desc=f"Copying labels for {split}"):
            lbl_name = os.path.splitext(img_name)[0] + '.txt'
            src_lbl_path = os.path.join(src_lbl_dir, lbl_name)
            if os.path.exists(src_lbl_path):
                shutil.copy2(src_lbl_path, os.path.join(tgt_lbl_dir, lbl_name))

    return selected_files

# ==========================================
# 3. MICRO-TRAIN GRID SEARCH ENGINE
# ==========================================
def run_micro_train_grid_search(source_base, sample_ratio=0.2, epochs=3):
    temp_base = "/content/micro_dataset_temp"
    selected_files = create_micro_dataset(source_base, temp_base, sample_ratio)

    class_names = {0: 'pothole'}

    temp_yaml_path = os.path.join(temp_base, "micro_data.yaml")
    with open(temp_yaml_path, 'w') as f:
        yaml.dump({
            'path': temp_base,
            'train': 'train/images',
            'val': 'valid/images',
            'names': class_names
        }, f)

    grids = {
        "CLAHE": {
            "func": apply_clahe,
            "param_names": ["clip_limit", "grid_size"],
            "combinations": list(itertools.product([1.0, 2.0, 3.0, 4.0], [(4, 4), (8, 8), (16, 16)]))
        },
        "AGCWD": {
            "func": apply_agcwd,
            "param_names": ["w"],
            "combinations": [(w,) for w in [0.1, 0.3, 0.5, 0.7, 0.9]]
        },
        "SSR": {
            "func": apply_ssr,
            "param_names": ["sigma"],
            "combinations": [(s,) for s in [15, 50, 80, 120, 250]]
        }
    }

    all_results = []
    print(f"\nStarting Micro-Training Grid Search ({epochs} Epochs per combination)...")

    for algo_name, config in grids.items():
        print(f"\n================ Tuning {algo_name} ================")

        for combo in config["combinations"]:
            params_dict = dict(zip(config["param_names"], combo))
            print(f"\n---> Testing {algo_name} with params: {params_dict}")

            # 1. Generate the Enhanced Images for this combination
            for split in ['train', 'valid']:
                src_img_dir = os.path.join(source_base, split, 'images')
                tgt_img_dir = os.path.join(temp_base, split, 'images')
                os.makedirs(tgt_img_dir, exist_ok=True)

                for img_name in selected_files[split]:
                    img = cv2.imread(os.path.join(src_img_dir, img_name))
                    enhanced = config["func"](img, **params_dict)
                    cv2.imwrite(os.path.join(tgt_img_dir, img_name), enhanced)

            # 2. Clear previous YOLO cache to prevent data overlap
            for split in ['train', 'valid']:
                cache_file = os.path.join(temp_base, split, 'labels.cache')
                if os.path.exists(cache_file): os.remove(cache_file)

            # 3. Initialize fresh Nano model and Micro-Train
            model = YOLO('yolov8n.pt')

            # plots=False saves time by not generating graphs
            results = model.train(
                data=temp_yaml_path,
                epochs=epochs,
                imgsz=640,
                batch=16,
                project='micro_tune_runs',
                name=f'{algo_name}_{combo}',
                save=False,
                plots=False,
                verbose=False
            )

            map50 = float(results.box.map50)
            print(f"{algo_name} {params_dict} | Micro-mAP50: {map50:.4f}")

            all_results.append({
                "Algorithm": algo_name,
                "Parameters": str(params_dict),
                "Micro_mAP50": round(map50, 4)
            })

            shutil.rmtree(os.path.join(temp_base, 'train', 'images'))
            shutil.rmtree(os.path.join(temp_base, 'valid', 'images'))

    # Save Results
    results_df = pd.DataFrame(all_results)
    csv_path = os.path.join(os.path.dirname(source_base), "Micro_Train_Grid_Search_Results.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"\nSaved Micro-Training Grid Search table to: {csv_path}")

    shutil.rmtree(temp_base, ignore_errors=True)

# ==========================================
# 4. EXECUTE
# ==========================================
SOURCE_BASE = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline"

run_micro_train_grid_search(
    source_base=SOURCE_BASE,
    sample_ratio=0.2,  # Uses 20% of dataset
    epochs=10          # The optimal threshold for micro-training
)



Creating 20.0% Micro-Dataset for fast tuning...


Copying labels for valid: 100%|██████████| 248/248 [03:31<00:00,  1.17it/s]



Starting Micro-Training Grid Search (10 Epochs per combination)...

================ Tuning CLAHE ================

---> Testing CLAHE with params: {'clip_limit': 1.0, 'grid_size': (4, 4)}
Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/micro_dataset_temp/micro_data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=

## Image Generation for Micro Training Grid Search

In [ ]:
import cv2
import os
import shutil
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. PURE MATHEMATICAL ALGORITHMS
# ==========================================e

def apply_clahe(img, clip_limit=1.5, grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

def apply_agcwd(img, w=0.3):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    v_channel = hsv[:, :, 2]
    hist, _ = np.histogram(v_channel.flatten(), 256, [0, 256])
    pdf = hist / np.sum(hist)
    max_pdf = np.max(pdf)
    min_pdf = np.min(pdf)
    if max_pdf == min_pdf: return img
    w_pdf = max_pdf * (((pdf - min_pdf) / (max_pdf - min_pdf)) ** w)
    w_cdf = np.cumsum(w_pdf) / np.sum(w_pdf)
    mapping = np.array([255 * (i / 255) ** (1 - w_cdf[i]) for i in range(256)], dtype=np.uint8)
    hsv[:, :, 2] = cv2.LUT(v_channel, mapping)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def apply_ssr(img, sigma=80):
    img_float = np.float64(img) + 1.0
    illumination = cv2.GaussianBlur(img_float, (0, 0), sigma)
    retinex = np.log10(img_float) - np.log10(illumination)
    enhanced = np.zeros_like(img)
    for i in range(3):
        channel = retinex[:, :, i]
        lower, upper = np.percentile(channel, (1, 99))
        channel_clipped = np.clip(channel, lower, upper)
        channel_norm = (channel_clipped - lower) / (upper - lower) * 255.0
        enhanced[:, :, i] = np.uint8(channel_norm)
    return enhanced

# ==========================================
# 2. METHODOLOGY-SAFE DATASET GENERATOR
# ==========================================

def generate_experimental_dataset(source_base, target_base, algo_func, algo_name, splits=['train', 'valid', 'dataset_split/test_night', 'dataset_split/test_day']):
    print(f"\nGenerating {algo_name} Dataset...")

    for split in splits:
        source_images = os.path.join(source_base, split, 'images')
        source_labels = os.path.join(source_base, split, 'labels')
        target_images = os.path.join(target_base, split, 'images')
        target_labels = os.path.join(target_base, split, 'labels')

        os.makedirs(target_images, exist_ok=True)

        # 1. Process Images
        if os.path.exists(source_images):
            image_files = [f for f in os.listdir(source_images) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            for img_name in tqdm(image_files, desc=f"Processing {split}"):
                img_path = os.path.join(source_images, img_name)
                img = cv2.imread(img_path)
                if img is None: continue

                enhanced_img = algo_func(img)
                cv2.imwrite(os.path.join(target_images, img_name), enhanced_img)

        # 2. Copy Labels Safely
        if os.path.exists(source_labels):
            os.makedirs(target_labels, exist_ok=True)
            for label_file in os.listdir(source_labels):
                if label_file.endswith('.txt'):
                    shutil.copy2(
                        os.path.join(source_labels, label_file),
                        os.path.join(target_labels, label_file)
                    )

# ==========================================
# 3. RUN THE GENERATOR
# ==========================================
SOURCE = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline"

generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '2_CLAHE_Tuned(1.5, [8,8])')}", apply_clahe, "CLAHE")
generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '3_AGCWD_Tuned(0.3)')}", apply_agcwd, "AGCWD")
generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '4_SSR_Tuned(80)')}", apply_ssr, "SSR")

print("\nALL EXPERIMENTAL DATASETS GENERATED SUCCESSFULLY!")


Generating CLAHE Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:13<00:00, 21.92it/s]



Generating AGCWD Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:09<00:00, 31.27it/s]



Generating SSR Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [03:44<00:00,  1.31it/s]



ALL EXPERIMENTAL DATASETS GENERATED SUCCESSFULLY!


## Probabilistic Domain Mixing

In [ ]:
import cv2
import os
import shutil
import numpy as np
import random
from tqdm import tqdm

# ================================
# 1. PURE MATHEMATICAL ALGORITHMS
# ================================

def apply_clahe(img, clip_limit=2.0, grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

def apply_agcwd(img, w=0.3):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    v_channel = hsv[:, :, 2]
    hist, _ = np.histogram(v_channel.flatten(), 256, [0, 256])
    pdf = hist / np.sum(hist)
    max_pdf = np.max(pdf)
    min_pdf = np.min(pdf)
    if max_pdf == min_pdf: return img
    w_pdf = max_pdf * (((pdf - min_pdf) / (max_pdf - min_pdf)) ** w)
    w_cdf = np.cumsum(w_pdf) / np.sum(w_pdf)
    mapping = np.array([255 * (i / 255) ** (1 - w_cdf[i]) for i in range(256)], dtype=np.uint8)
    hsv[:, :, 2] = cv2.LUT(v_channel, mapping)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def apply_ssr(img, sigma=50):
    img_float = np.float64(img) + 1.0
    illumination = cv2.GaussianBlur(img_float, (0, 0), sigma)
    retinex = np.log10(img_float) - np.log10(illumination)
    enhanced = np.zeros_like(img)
    for i in range(3):
        channel = retinex[:, :, i]
        lower, upper = np.percentile(channel, (1, 99))
        if upper == lower: continue
        channel_clipped = np.clip(channel, lower, upper)
        channel_norm = (channel_clipped - lower) / (upper - lower) * 255.0
        enhanced[:, :, i] = np.uint8(channel_norm)
    return enhanced

# ==========================================
# 2. METHODOLOGY-SAFE DATASET GENERATOR
# ==========================================

def generate_experimental_dataset(source_base, target_base, algo_func, algo_name, mix_train=False, splits=['train', 'valid', 'dataset_split/test_night', 'dataset_split/test_day']):
    print(f"\nGenerating {algo_name} Dataset (Domain Mix: {mix_train})...")

    for split in splits:
        source_images = os.path.join(source_base, split, 'images')
        source_labels = os.path.join(source_base, split, 'labels')
        target_images = os.path.join(target_base, split, 'images')
        target_labels = os.path.join(target_base, split, 'labels')

        os.makedirs(target_images, exist_ok=True)

        # 1. Process Images with Probabilistic Mixing
        if os.path.exists(source_images):
            image_files = [f for f in os.listdir(source_images) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            for img_name in tqdm(image_files, desc=f"Processing {split}"):
                img_path = os.path.join(source_images, img_name)
                img = cv2.imread(img_path)
                if img is None: continue

                # Domain Mixing Logic for the Training Split
                if split == 'train' and mix_train:
                    if random.random() < 0.5:
                        final_img = algo_func(img)  # 50% chance: Apply Enhancement
                    else:
                        final_img = img             # 50% chance: Keep raw Baseline image
                else:
                    # For valid and test splits, apply enhancement 100% of the time
                    final_img = algo_func(img)

                cv2.imwrite(os.path.join(target_images, img_name), final_img)

        # 2. Copy Labels Safely
        if os.path.exists(source_labels):
            os.makedirs(target_labels, exist_ok=True)
            for label_file in os.listdir(source_labels):
                if label_file.endswith('.txt'):
                    shutil.copy2(
                        os.path.join(source_labels, label_file),
                        os.path.join(target_labels, label_file)
                    )

# ==========================================
# 3. RUN THE GENERATOR
# ==========================================
SOURCE = "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline"

# Generate Mixed Datasets for all three algorithms with mix_train=True
generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '2_CLAHE_Mixed')}", apply_clahe, "CLAHE_Mixed", mix_train=True)
generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '3_AGCWD_Mixed')}", apply_agcwd, "AGCWD_Mixed", mix_train=True)
generate_experimental_dataset(SOURCE, f"{SOURCE.replace('1_Baseline', '4_SSR_Mixed')}", apply_ssr, "SSR_Mixed", mix_train=True)

print("\nALL PROBABILISTIC MIXED DATASETS GENERATED SUCCESSFULLY!")


Generating CLAHE_Mixed Dataset (Domain Mix: True)...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:15<00:00, 18.95it/s]



Generating AGCWD_Mixed Dataset (Domain Mix: True)...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:09<00:00, 29.91it/s]



Generating SSR_Mixed Dataset (Domain Mix: True)...


Processing dataset_split/test_day: 100%|██████████| 294/294 [02:42<00:00,  1.81it/s]



ALL PROBABILISTIC MIXED DATASETS GENERATED SUCCESSFULLY!


## CLAHE Blending (0.7)

In [ ]:

import cv2
import os
import shutil
from tqdm import tqdm

# ==========================================
# CLAHE WITH WEIGHTED BLENDING
# ==========================================

def apply_clahe_blended(img, clip_limit=2.0, grid_size=(8, 8), alpha=0.7):
    """
    alpha = 0.7 means:
        70% CLAHE-enhanced luminance
        30% Original luminance
    """

    # Validate blending weight
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("Alpha must be between 0 and 1.")

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    enhanced_l = clahe.apply(l)

    # Blend enhanced luminance with original luminance
    blended_l = cv2.addWeighted(enhanced_l, alpha, l, 1.0 - alpha, 0)

    # Merge the blended L channel with original A and B
    merged = cv2.merge((blended_l, a, b))

    final_img = cv2.cvtColor(merged,cv2.COLOR_LAB2BGR)

    return final_img

# ==========================================
# DATASET GENERATOR
# ==========================================

def generate_experimental_dataset(source_base,target_base,algo_func,algo_name,splits=('train','valid','dataset_split/test_night','dataset_split/test_day')):
    print(f"\nGenerating {algo_name} Dataset...")

    total_images = 0
    total_labels = 0

    for split in splits:
        source_images = os.path.join(source_base, split, 'images')
        source_labels = os.path.join(source_base, split, 'labels')
        target_images = os.path.join(target_base, split, 'images')
        target_labels = os.path.join(target_base, split, 'labels')

        os.makedirs(target_images, exist_ok=True)

        # ======================================
        # PROCESS IMAGES
        # ======================================

        if os.path.exists(source_images):

            image_files = sorted([f for f in os.listdir(source_images) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

            for img_name in tqdm(image_files,desc=f"Processing {split}"):
                img_path = os.path.join(source_images, img_name)
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Warning: Cannot read {img_path}")
                    continue

                # Apply CLAHE weighted blending
                enhanced_img = algo_func(img)

                # Save enhanced image
                output_path = os.path.join(target_images, img_name)
                success = cv2.imwrite(output_path, enhanced_img)
                if success:
                    total_images += 1
                else:
                    print(f"Warning: Cannot save {output_path}")

        else:
            print(f"Warning: Missing image directory: {source_images}")

        # ======================================
        # COPY YOLO LABELS
        # ======================================

        if os.path.exists(source_labels):
            os.makedirs(target_labels, exist_ok=True)
            label_files = [ f for f in os.listdir(source_labels) if f.lower().endswith('.txt')]
            for label_file in label_files:
                shutil.copy2(
                    os.path.join(source_labels, label_file),
                    os.path.join(target_labels, label_file))
                total_labels += 1
        else:
            print(f"Warning: Missing label directory: {source_labels}")

    # ==========================================
    # GENERATION SUMMARY
    # ==========================================

    print(f"\n{algo_name} Dataset Generation Completed!")
    print(f"Total images processed: {total_images}")
    print(f"Total labels copied: {total_labels}")
    print(f"Output directory: {target_base}")

# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================

# Original baseline dataset
SOURCE = ("/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline")

# Output dataset
TARGET = ("/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend")

# CLAHE hyperparameters
CLIP_LIMIT = 2.0
GRID_SIZE = (8, 8)

# Weighted blending parameter
ALPHA = 0.7

# ==========================================
# RUN DATASET GENERATION
# ==========================================

generate_experimental_dataset(
    source_base=SOURCE,
    target_base=TARGET,
    algo_func=lambda img: apply_clahe_blended(
        img,
        clip_limit=CLIP_LIMIT,
        grid_size=GRID_SIZE,
        alpha=ALPHA
    ),
    algo_name="CLAHE Weighted Blending"
)

print("\nALL CLAHE BLENDED DATASETS GENERATED SUCCESSFULLY!")



Generating CLAHE Weighted Blending Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:18<00:00, 15.89it/s]



CLAHE Weighted Blending Dataset Generation Completed!
Total images processed: 6084
Total labels copied: 6084
Output directory: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend

ALL CLAHE BLENDED DATASETS GENERATED SUCCESSFULLY!


## CLAHE Blending (0.8)

In [ ]:

import cv2
import os
import shutil
from tqdm import tqdm

# ==========================================
# CLAHE WITH WEIGHTED BLENDING
# ==========================================

def apply_clahe_blended(img, clip_limit=2.0, grid_size=(8, 8), alpha=0.8):
    # Validate blending weight
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("Alpha must be between 0 and 1.")

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    enhanced_l = clahe.apply(l)

    # Blend enhanced luminance with original luminance
    blended_l = cv2.addWeighted(enhanced_l, alpha, l, 1.0 - alpha, 0)

    # Merge the blended L channel with original A and B
    merged = cv2.merge((blended_l, a, b))

    final_img = cv2.cvtColor(merged,cv2.COLOR_LAB2BGR)

    return final_img

# ==========================================
# DATASET GENERATOR
# ==========================================

def generate_experimental_dataset(source_base,target_base,algo_func,algo_name,splits=('train','valid','dataset_split/test_night','dataset_split/test_day')):
    print(f"\nGenerating {algo_name} Dataset...")

    total_images = 0
    total_labels = 0

    for split in splits:
        source_images = os.path.join(source_base, split, 'images')
        source_labels = os.path.join(source_base, split, 'labels')
        target_images = os.path.join(target_base, split, 'images')
        target_labels = os.path.join(target_base, split, 'labels')

        os.makedirs(target_images, exist_ok=True)

        # ======================================
        # PROCESS IMAGES
        # ======================================

        if os.path.exists(source_images):

            image_files = sorted([f for f in os.listdir(source_images) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

            for img_name in tqdm(image_files,desc=f"Processing {split}"):
                img_path = os.path.join(source_images, img_name)
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Warning: Cannot read {img_path}")
                    continue

                # Apply CLAHE weighted blending
                enhanced_img = algo_func(img)

                # Save enhanced image
                output_path = os.path.join(target_images, img_name)
                success = cv2.imwrite(output_path, enhanced_img)
                if success:
                    total_images += 1
                else:
                    print(f"Warning: Cannot save {output_path}")

        else:
            print(f"Warning: Missing image directory: {source_images}")

        # ======================================
        # COPY YOLO LABELS
        # ======================================

        if os.path.exists(source_labels):
            os.makedirs(target_labels, exist_ok=True)
            label_files = [ f for f in os.listdir(source_labels) if f.lower().endswith('.txt')]
            for label_file in label_files:
                shutil.copy2(
                    os.path.join(source_labels, label_file),
                    os.path.join(target_labels, label_file))
                total_labels += 1
        else:
            print(f"Warning: Missing label directory: {source_labels}")

    # ==========================================
    # GENERATION SUMMARY
    # ==========================================

    print(f"\n{algo_name} Dataset Generation Completed!")
    print(f"Total images processed: {total_images}")
    print(f"Total labels copied: {total_labels}")
    print(f"Output directory: {target_base}")

# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================

# Original baseline dataset
SOURCE = ("/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline")

# Output dataset
TARGET = ("/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend(0.8)")

# CLAHE hyperparameters
CLIP_LIMIT = 2.0
GRID_SIZE = (8, 8)

# Weighted blending parameter
ALPHA = 0.8

# ==========================================
# RUN DATASET GENERATION
# ==========================================

generate_experimental_dataset(
    source_base=SOURCE,
    target_base=TARGET,
    algo_func=lambda img: apply_clahe_blended(
        img,
        clip_limit=CLIP_LIMIT,
        grid_size=GRID_SIZE,
        alpha=ALPHA
    ),
    algo_name="CLAHE Weighted Blending (0.8)"
)

print("\nALL CLAHE BLENDED DATASETS GENERATED SUCCESSFULLY!")



Generating CLAHE Weighted Blending (0.8) Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:16<00:00, 17.60it/s]



CLAHE Weighted Blending (0.8) Dataset Generation Completed!
Total images processed: 6084
Total labels copied: 6084
Output directory: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend(0.8)

ALL CLAHE BLENDED DATASETS GENERATED SUCCESSFULLY!


## CLAHE Blending (0.5)

In [5]:

import cv2
import os
import shutil
from tqdm import tqdm

# ==========================================
# CLAHE WITH WEIGHTED BLENDING
# ==========================================

def apply_clahe_blended(img, clip_limit=2.0, grid_size=(8, 8), alpha=0.8):
    # Validate blending weight
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("Alpha must be between 0 and 1.")

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    enhanced_l = clahe.apply(l)

    # Blend enhanced luminance with original luminance
    blended_l = cv2.addWeighted(enhanced_l, alpha, l, 1.0 - alpha, 0)

    # Merge the blended L channel with original A and B
    merged = cv2.merge((blended_l, a, b))

    final_img = cv2.cvtColor(merged,cv2.COLOR_LAB2BGR)

    return final_img

# ==========================================
# DATASET GENERATOR
# ==========================================

def generate_experimental_dataset(source_base,target_base,algo_func,algo_name,splits=('train','valid','dataset_split/test_night','dataset_split/test_day')):
    print(f"\nGenerating {algo_name} Dataset...")

    total_images = 0
    total_labels = 0

    for split in splits:
        source_images = os.path.join(source_base, split, 'images')
        source_labels = os.path.join(source_base, split, 'labels')
        target_images = os.path.join(target_base, split, 'images')
        target_labels = os.path.join(target_base, split, 'labels')

        os.makedirs(target_images, exist_ok=True)

        # ======================================
        # PROCESS IMAGES
        # ======================================

        if os.path.exists(source_images):

            image_files = sorted([f for f in os.listdir(source_images) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

            for img_name in tqdm(image_files,desc=f"Processing {split}"):
                img_path = os.path.join(source_images, img_name)
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Warning: Cannot read {img_path}")
                    continue

                # Apply CLAHE weighted blending
                enhanced_img = algo_func(img)

                # Save enhanced image
                output_path = os.path.join(target_images, img_name)
                success = cv2.imwrite(output_path, enhanced_img)
                if success:
                    total_images += 1
                else:
                    print(f"Warning: Cannot save {output_path}")

        else:
            print(f"Warning: Missing image directory: {source_images}")

        # ======================================
        # COPY YOLO LABELS
        # ======================================

        if os.path.exists(source_labels):
            os.makedirs(target_labels, exist_ok=True)
            label_files = [ f for f in os.listdir(source_labels) if f.lower().endswith('.txt')]
            for label_file in label_files:
                shutil.copy2(
                    os.path.join(source_labels, label_file),
                    os.path.join(target_labels, label_file))
                total_labels += 1
        else:
            print(f"Warning: Missing label directory: {source_labels}")

    # ==========================================
    # GENERATION SUMMARY
    # ==========================================

    print(f"\n{algo_name} Dataset Generation Completed!")
    print(f"Total images processed: {total_images}")
    print(f"Total labels copied: {total_labels}")
    print(f"Output directory: {target_base}")

# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================

# Original baseline dataset
SOURCE = ("/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline")

# Output dataset
TARGET = ("/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend(0.5)")

# CLAHE hyperparameters
CLIP_LIMIT = 2.0
GRID_SIZE = (8, 8)

# Weighted blending parameter
ALPHA = 0.5

# ==========================================
# RUN DATASET GENERATION
# ==========================================

generate_experimental_dataset(
    source_base=SOURCE,
    target_base=TARGET,
    algo_func=lambda img: apply_clahe_blended(
        img,
        clip_limit=CLIP_LIMIT,
        grid_size=GRID_SIZE,
        alpha=ALPHA
    ),
    algo_name="CLAHE Weighted Blending (0.5)"
)

print("\nALL CLAHE BLENDED DATASETS GENERATED SUCCESSFULLY!")



Generating CLAHE Weighted Blending (0.5) Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [01:35<00:00,  3.07it/s]



CLAHE Weighted Blending (0.5) Dataset Generation Completed!
Total images processed: 6084
Total labels copied: 6084
Output directory: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend(0.5)

ALL CLAHE BLENDED DATASETS GENERATED SUCCESSFULLY!


## CLAHE Blend (0.6 , 0.9)

In [ ]:
import cv2
import os
import shutil
from tqdm import tqdm

# ==========================================
# CLAHE WITH WEIGHTED BLENDING
# ==========================================

def apply_clahe_blended(img, clip_limit=2.0, grid_size=(8, 8), alpha=0.7):
    # Validate blending weight
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("Alpha must be between 0 and 1.")

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    enhanced_l = clahe.apply(l)

    # Blend enhanced luminance with original luminance
    blended_l = cv2.addWeighted(enhanced_l, alpha, l, 1.0 - alpha, 0)

    # Merge the blended L channel with original A and B
    merged = cv2.merge((blended_l, a, b))

    final_img = cv2.cvtColor(merged,cv2.COLOR_LAB2BGR)

    return final_img

# ==========================================
# DATASET GENERATOR
# ==========================================

def generate_experimental_dataset(source_base,target_base,algo_func,algo_name,splits=('train','valid','dataset_split/test_night','dataset_split/test_day')):
    print(f"\nGenerating {algo_name} Dataset...")

    total_images = 0
    total_labels = 0

    for split in splits:
        source_images = os.path.join(source_base, split, 'images')
        source_labels = os.path.join(source_base, split, 'labels')
        target_images = os.path.join(target_base, split, 'images')
        target_labels = os.path.join(target_base, split, 'labels')

        os.makedirs(target_images, exist_ok=True)

        # ======================================
        # PROCESS IMAGES
        # ======================================

        if os.path.exists(source_images):

            image_files = sorted([f for f in os.listdir(source_images) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

            for img_name in tqdm(image_files,desc=f"Processing {split}"):
                img_path = os.path.join(source_images, img_name)
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Warning: Cannot read {img_path}")
                    continue

                # Apply CLAHE weighted blending
                enhanced_img = algo_func(img)

                # Save enhanced image
                output_path = os.path.join(target_images, img_name)
                success = cv2.imwrite(output_path, enhanced_img)
                if success:
                    total_images += 1
                else:
                    print(f"Warning: Cannot save {output_path}")

        else:
            print(f"Warning: Missing image directory: {source_images}")

        # ======================================
        # COPY YOLO LABELS
        # ======================================

        if os.path.exists(source_labels):
            os.makedirs(target_labels, exist_ok=True)
            label_files = [ f for f in os.listdir(source_labels) if f.lower().endswith('.txt')]
            for label_file in label_files:
                shutil.copy2(
                    os.path.join(source_labels, label_file),
                    os.path.join(target_labels, label_file))
                total_labels += 1
        else:
            print(f"Warning: Missing label directory: {source_labels}")

    # ==========================================
    # GENERATION SUMMARY
    # ==========================================

    print(f"\n{algo_name} Dataset Generation Completed!")
    print(f"Total images processed: {total_images}")
    print(f"Total labels copied: {total_labels}")
    print(f"Output directory: {target_base}")

# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================

# Original baseline dataset
SOURCE = ("/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline")

# CLAHE hyperparameters
CLIP_LIMIT = 2.0
GRID_SIZE = (8, 8)

# Weighted blending parameters to iterate over
alphas = [0.6, 0.9]

# ==========================================
# RUN DATASET GENERATION
# ==========================================

for alpha in alphas:
    TARGET = f"/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend({alpha})"

    generate_experimental_dataset(
        source_base=SOURCE,
        target_base=TARGET,
        algo_func=lambda img, a=alpha: apply_clahe_blended(
            img,
            clip_limit=CLIP_LIMIT,
            grid_size=GRID_SIZE,
            alpha=a
        ),
        algo_name=f"CLAHE Weighted Blending ({alpha})"
    )

print("\nALL CLAHE BLENDED DATASETS GENERATED SUCCESSFULLY!")


Generating CLAHE Weighted Blending (0.6) Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:16<00:00, 17.85it/s]



CLAHE Weighted Blending (0.6) Dataset Generation Completed!
Total images processed: 6084
Total labels copied: 6084
Output directory: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend(0.6)

Generating CLAHE Weighted Blending (0.9) Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:13<00:00, 21.56it/s]



CLAHE Weighted Blending (0.9) Dataset Generation Completed!
Total images processed: 6084
Total labels copied: 6084
Output directory: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_2_CLAHE_Blend(0.9)

ALL CLAHE BLENDED DATASETS GENERATED SUCCESSFULLY!


## AGCWD Blend (0.6, 0.7, 0.8)

In [ ]:

import cv2
import os
import shutil
import numpy as np
from tqdm import tqdm

# ==========================================
# AGCWD WITH WEIGHTED BLENDING
# ==========================================

def apply_agcwd_blended(img, w=0.3, alpha=0.7):

    # Validate blending weight
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("Alpha must be between 0 and 1.")

    # Convert BGR to HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Extract original Value channel
    v_channel = hsv[:, :, 2]

    # Calculate histogram
    hist, _ = np.histogram(v_channel.flatten(), 256, [0, 256])

    # Calculate probability distribution function (PDF)
    pdf = hist / np.sum(hist)

    max_pdf = np.max(pdf)
    min_pdf = np.min(pdf)

    # Handle uniform histogram
    if max_pdf == min_pdf:
        return img

    # Calculate weighted probability distribution
    w_pdf = max_pdf * (
        ((pdf - min_pdf) / (max_pdf - min_pdf)) ** w
    )

    # Calculate weighted cumulative distribution
    w_cdf = np.cumsum(w_pdf) / np.sum(w_pdf)

    # Generate intensity mapping
    mapping = np.array(
        [
            255 * (i / 255) ** (1 - w_cdf[i])
            for i in range(256)
        ],
        dtype=np.uint8
    )

    # Apply AGCWD mapping
    enhanced_v = cv2.LUT(v_channel, mapping)

    # ======================================
    # WEIGHTED BLENDING
    # ======================================

    # Blend enhanced V channel with original V channel
    blended_v = cv2.addWeighted(
        enhanced_v, alpha,
        v_channel, 1.0 - alpha,
        0
    )

    # Replace original V channel with blended V channel
    hsv[:, :, 2] = blended_v

    # Convert HSV back to BGR
    final_img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    return final_img


# ==========================================
# DATASET GENERATOR
# ==========================================

def generate_experimental_dataset(
    source_base,
    target_base,
    algo_func,
    algo_name,
    splits=('train', 'valid', 'dataset_split/test_night', 'dataset_split/test_day')
):

    print(f"\nGenerating {algo_name} Dataset...")

    total_images = 0
    total_labels = 0

    for split in splits:
        source_images = os.path.join(source_base, split, 'images')
        source_labels = os.path.join(source_base, split, 'labels')
        target_images = os.path.join(target_base, split, 'images')
        target_labels = os.path.join(target_base, split, 'labels')
        os.makedirs(target_images, exist_ok=True)

        # ======================================
        # PROCESS IMAGES
        # ======================================

        if os.path.exists(source_images):
            image_files = sorted([
                f for f in os.listdir(source_images)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            ])

            for img_name in tqdm(image_files, desc=f"Processing {split}"):
                img_path = os.path.join(source_images, img_name)
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Warning: Cannot read {img_path}")
                    continue

                # Apply AGCWD weighted blending
                enhanced_img = algo_func(img)
                # Save enhanced image
                output_path = os.path.join(target_images, img_name)
                success = cv2.imwrite(output_path, enhanced_img)
                if success:
                    total_images += 1
                else:
                    print(f"Warning: Cannot save {output_path}")

        else:
            print(f"Warning: Missing image directory: {source_images}")

        # ======================================
        # COPY YOLO LABELS
        # ======================================

        if os.path.exists(source_labels):
            os.makedirs(target_labels, exist_ok=True)
            label_files = sorted([
                f for f in os.listdir(source_labels)
                if f.lower().endswith('.txt')
            ])
            for label_file in label_files:
                shutil.copy2(
                    os.path.join(source_labels, label_file),
                    os.path.join(target_labels, label_file)
                )
                total_labels += 1

        else:
            print(f"Warning: Missing label directory: {source_labels}")

    # ==========================================
    # GENERATION SUMMARY
    # ==========================================

    print(f"\n{algo_name} Dataset Generation Completed!")
    print(f"Total images processed: {total_images}")
    print(f"Total labels copied: {total_labels}")
    print(f"Output directory: {target_base}")


# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================

# Original baseline dataset
SOURCE = (
    "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline"
)

# AGCWD hyperparameter
W = 0.3

# Weighted blending parameters to iterate over
alphas = [0.6, 0.7, 0.8]


# ==========================================
# RUN DATASET GENERATION
# ==========================================

for alpha in alphas:

    TARGET = (
        f"/content/drive/MyDrive/Thesis_Pothole_Project/"
        f"Dataset_3_AGCWD_Blend({alpha})"
    )

    generate_experimental_dataset(
        source_base=SOURCE,
        target_base=TARGET,
        algo_func=lambda img, a=alpha: apply_agcwd_blended(
            img,
            w=W,
            alpha=a
        ),
        algo_name=f"AGCWD Weighted Blending ({alpha})"
    )

print("\nALL AGCWD BLENDED DATASETS GENERATED SUCCESSFULLY!")



Generating AGCWD Weighted Blending (0.6) Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:16<00:00, 18.33it/s]



AGCWD Weighted Blending (0.6) Dataset Generation Completed!
Total images processed: 6084
Total labels copied: 6084
Output directory: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_3_AGCWD_Blend(0.6)

Generating AGCWD Weighted Blending (0.7) Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:12<00:00, 22.83it/s]



AGCWD Weighted Blending (0.7) Dataset Generation Completed!
Total images processed: 6084
Total labels copied: 6084
Output directory: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_3_AGCWD_Blend(0.7)

Generating AGCWD Weighted Blending (0.8) Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:15<00:00, 19.39it/s]



AGCWD Weighted Blending (0.8) Dataset Generation Completed!
Total images processed: 6084
Total labels copied: 6084
Output directory: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_3_AGCWD_Blend(0.8)

ALL AGCWD BLENDED DATASETS GENERATED SUCCESSFULLY!


## AGCWD Blend 0.5

In [ ]:

import cv2
import os
import shutil
import numpy as np
from tqdm import tqdm

# ==========================================
# AGCWD WITH WEIGHTED BLENDING
# ==========================================

def apply_agcwd_blended(img, w=0.3, alpha=0.7):

    # Validate blending weight
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("Alpha must be between 0 and 1.")

    # Convert BGR to HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Extract original Value channel
    v_channel = hsv[:, :, 2]

    # Calculate histogram
    hist, _ = np.histogram(v_channel.flatten(), 256, [0, 256])

    # Calculate probability distribution function (PDF)
    pdf = hist / np.sum(hist)

    max_pdf = np.max(pdf)
    min_pdf = np.min(pdf)

    # Handle uniform histogram
    if max_pdf == min_pdf:
        return img

    # Calculate weighted probability distribution
    w_pdf = max_pdf * (
        ((pdf - min_pdf) / (max_pdf - min_pdf)) ** w
    )

    # Calculate weighted cumulative distribution
    w_cdf = np.cumsum(w_pdf) / np.sum(w_pdf)

    # Generate intensity mapping
    mapping = np.array(
        [
            255 * (i / 255) ** (1 - w_cdf[i])
            for i in range(256)
        ],
        dtype=np.uint8
    )

    # Apply AGCWD mapping
    enhanced_v = cv2.LUT(v_channel, mapping)

    # ======================================
    # WEIGHTED BLENDING
    # ======================================

    # Blend enhanced V channel with original V channel
    blended_v = cv2.addWeighted(
        enhanced_v, alpha,
        v_channel, 1.0 - alpha,
        0
    )

    # Replace original V channel with blended V channel
    hsv[:, :, 2] = blended_v

    # Convert HSV back to BGR
    final_img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    return final_img


# ==========================================
# DATASET GENERATOR
# ==========================================

def generate_experimental_dataset(
    source_base,
    target_base,
    algo_func,
    algo_name,
    splits=('train', 'valid', 'dataset_split/test_night', 'dataset_split/test_day')
):

    print(f"\nGenerating {algo_name} Dataset...")

    total_images = 0
    total_labels = 0

    for split in splits:
        source_images = os.path.join(source_base, split, 'images')
        source_labels = os.path.join(source_base, split, 'labels')
        target_images = os.path.join(target_base, split, 'images')
        target_labels = os.path.join(target_base, split, 'labels')
        os.makedirs(target_images, exist_ok=True)

        # ======================================
        # PROCESS IMAGES
        # ======================================

        if os.path.exists(source_images):
            image_files = sorted([
                f for f in os.listdir(source_images)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            ])

            for img_name in tqdm(image_files, desc=f"Processing {split}"):
                img_path = os.path.join(source_images, img_name)
                img = cv2.imread(img_path)

                if img is None:
                    print(f"Warning: Cannot read {img_path}")
                    continue

                # Apply AGCWD weighted blending
                enhanced_img = algo_func(img)
                # Save enhanced image
                output_path = os.path.join(target_images, img_name)
                success = cv2.imwrite(output_path, enhanced_img)

                if success:
                    total_images += 1
                else:
                    print(f"Warning: Cannot save {output_path}")

        else:
            print(f"Warning: Missing image directory: {source_images}")

        # ======================================
        # COPY YOLO LABELS
        # ======================================

        if os.path.exists(source_labels):
            os.makedirs(target_labels, exist_ok=True)
            label_files = sorted([
                f for f in os.listdir(source_labels)
                if f.lower().endswith('.txt')
            ])

            for label_file in label_files:
                shutil.copy2(
                    os.path.join(source_labels, label_file),
                    os.path.join(target_labels, label_file)
                )
                total_labels += 1

        else:
            print(f"Warning: Missing label directory: {source_labels}")

    # ==========================================
    # GENERATION SUMMARY
    # ==========================================

    print(f"\n{algo_name} Dataset Generation Completed!")
    print(f"Total images processed: {total_images}")
    print(f"Total labels copied: {total_labels}")
    print(f"Output directory: {target_base}")


# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================

# Original baseline dataset
SOURCE = (
    "/content/drive/MyDrive/Thesis_Pothole_Project/Dataset_1_Baseline"
)

# AGCWD hyperparameter
W = 0.3

# Weighted blending parameters to iterate over
alphas = [0.5]


# ==========================================
# RUN DATASET GENERATION
# ==========================================

for alpha in alphas:

    TARGET = (
        f"/content/drive/MyDrive/Thesis_Pothole_Project/"
        f"Dataset_3_AGCWD_Blend({alpha})"
    )

    generate_experimental_dataset(
        source_base=SOURCE,
        target_base=TARGET,
        algo_func=lambda img, a=alpha: apply_agcwd_blended(
            img,
            w=W,
            alpha=a
        ),
        algo_name=f"AGCWD Weighted Blending ({alpha})"
    )

print("\nALL AGCWD BLENDED DATASETS GENERATED SUCCESSFULLY!")



Generating AGCWD Weighted Blending (0.5) Dataset...


Processing dataset_split/test_day: 100%|██████████| 294/294 [00:17<00:00, 17.22it/s]



AGCWD Weighted Blending (0.5) Dataset Generation Completed!
Total images processed: 6084
Total labels copied: 6084
Output directory: /content/drive/MyDrive/Thesis_Pothole_Project/Dataset_3_AGCWD_Blend(0.5)

ALL AGCWD BLENDED DATASETS GENERATED SUCCESSFULLY!
